# Construção da Camada Gold

## Objetivo

A camada Gold tem como objetivo disponibilizar dados refinados e organizados para consumo analítico, utilizando como fonte os dados previamente tratados e validados na camada Silver.

Nesta etapa, os dados da tabela `workspace.silver.ibge_pam_cafe` serão utilizados para construir estruturas voltadas às análises do MVP, permitindo responder às perguntas definidas no projeto sobre a produção de café no Brasil entre 2015 e 2025.

A camada Gold será construída a partir de agregações e transformações realizadas sobre os dados da Silver, preservando a rastreabilidade do fluxo de dados entre as camadas Bronze, Silver e Gold.

A construção da camada Gold será orientada pelas seguintes perguntas de negócio definidas para o MVP:

1. Quais estados e regiões concentram a maior produção de café?
2. Como a produção de café evoluiu ao longo do período analisado?
3. Quais estados apresentam maior produtividade?
4. Como a participação dos principais estados produtores mudou ao longo dos anos?
5. Como a produção de café arábica e conilon se distribui entre os principais estados produtores?

As estruturas analíticas da camada Gold serão construídas a partir dos dados tratados da camada Silver, de forma a permitir responder a essas perguntas por meio de consultas, agregações e visualizações.

In [0]:
# Criação do schema da camada Gold

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold
COMMENT 'Camada Gold do MVP de Engenharia de Dados - dados agregados e preparados para análise de negócio'
""")

print("Schema workspace.gold criado com sucesso.")

In [0]:
# Leitura dos dados da camada Silver

df_silver = spark.table("workspace.silver.ibge_pam_cafe")

print("Quantidade de registros carregados da Silver:", df_silver.count())

print("\nSchema:")
df_silver.printSchema()

In [0]:
# Verificação das Unidades da Federação presentes na Silver

ufs_silver = (
    df_silver
    .select("uf")
    .distinct()
    .orderBy("uf")
)

print("Quantidade de UFs distintas:", ufs_silver.count())

display(ufs_silver)

In [0]:
# Verificação dos registros de Guanabara antes da preparação da Gold

df_guanabara = (
    df_silver
    .filter(df_silver.uf == "Guanabara")
)

print("Total de registros de Guanabara:", df_guanabara.count())

print(
    "Registros de Guanabara com valor numérico:",
    df_guanabara.filter(df_guanabara.valor.isNotNull()).count()
)

display(
    df_guanabara
    .groupBy("variavel")
    .count()
    .orderBy("variavel")
)

### Preparação dos dados para análise

Antes da construção das estruturas analíticas da camada Gold, foi realizada uma verificação das Unidades da Federação presentes na camada Silver.

Foram identificados 28 registros territoriais distintos, incluindo Guanabara. A análise dos dados mostrou que Guanabara possui 220 registros no conjunto, porém nenhum deles apresenta valor numérico válido para o período analisado (2015 a 2025).

Dessa forma, Guanabara será mantida na camada Silver para preservar a fidelidade e a rastreabilidade em relação à fonte original, mas será desconsiderada nas análises da camada Gold.

A camada Gold utilizará, portanto, as 27 Unidades da Federação atuais do Brasil para a construção das análises por estado e região.

In [0]:
# Criação da base analítica da camada Gold
# Guanabara é mantida na Silver, mas desconsiderada nas análises da Gold

df_gold_base = (
    df_silver
    .filter(df_silver.uf != "Guanabara")
)

print("Registros na Silver:", df_silver.count())
print("Registros na base Gold:", df_gold_base.count())

print(
    "Quantidade de UFs na base Gold:",
    df_gold_base.select("uf").distinct().count()
)

print(
    "Registros de Guanabara na base Gold:",
    df_gold_base.filter(df_gold_base.uf == "Guanabara").count()
)

In [0]:
# Mapeamento das Unidades da Federação para as regiões do Brasil

mapa_regioes = {
    # Norte
    "Acre": "Norte",
    "Amapá": "Norte",
    "Amazonas": "Norte",
    "Pará": "Norte",
    "Rondônia": "Norte",
    "Roraima": "Norte",
    "Tocantins": "Norte",

    # Nordeste
    "Alagoas": "Nordeste",
    "Bahia": "Nordeste",
    "Ceará": "Nordeste",
    "Maranhão": "Nordeste",
    "Paraíba": "Nordeste",
    "Pernambuco": "Nordeste",
    "Piauí": "Nordeste",
    "Rio Grande do Norte": "Nordeste",
    "Sergipe": "Nordeste",

    # Centro-Oeste
    "Distrito Federal": "Centro-Oeste",
    "Goiás": "Centro-Oeste",
    "Mato Grosso": "Centro-Oeste",
    "Mato Grosso do Sul": "Centro-Oeste",

    # Sudeste
    "Espírito Santo": "Sudeste",
    "Minas Gerais": "Sudeste",
    "Rio de Janeiro": "Sudeste",
    "São Paulo": "Sudeste",

    # Sul
    "Paraná": "Sul",
    "Rio Grande do Sul": "Sul",
    "Santa Catarina": "Sul"
}

print("Quantidade de UFs no mapeamento:", len(mapa_regioes))

for regiao in ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]:
    quantidade = sum(1 for r in mapa_regioes.values() if r == regiao)
    print(f"{regiao}: {quantidade} UFs")

In [0]:
# Adicionando a região geográfica à base Gold

from pyspark.sql.functions import create_map, lit, col
from itertools import chain

mapa_regioes_spark = create_map(
    [lit(x) for x in chain(*mapa_regioes.items())]
)

df_gold_base = (
    df_gold_base
    .withColumn(
        "regiao",
        mapa_regioes_spark[col("uf")]
    )
)

print("Total de registros:", df_gold_base.count())

print(
    "Registros sem região:",
    df_gold_base.filter(col("regiao").isNull()).count()
)

print(
    "Quantidade de regiões:",
    df_gold_base.select("regiao").distinct().count()
)

display(
    df_gold_base
    .select("uf", "regiao")
    .distinct()
    .orderBy("regiao", "uf")
)

### Enriquecimento com região geográfica

Para permitir análises regionais, foi adicionada à base Gold a dimensão `regiao`, associando cada uma das 27 Unidades da Federação atuais às cinco regiões brasileiras: Norte, Nordeste, Centro-Oeste, Sudeste e Sul.

Após o enriquecimento, foi realizada uma validação para garantir que todas as UFs utilizadas na camada Gold possuíssem uma região correspondente.

A base analítica permaneceu com 5.940 registros e não foram identificados registros sem classificação regional.

Essa informação permitirá realizar agregações tanto por Unidade da Federação quanto por região geográfica nas análises do MVP.

In [0]:
# Comparação entre "Total" e "Café (em grão) Total"

from pyspark.sql.functions import col, count

comparacao_produtos = (
    df_gold_base
    .filter(
        col("produto").isin(
            "Total",
            "Café (em grão) Total"
        )
    )
    .groupBy("variavel", "produto")
    .agg(
        count("*").alias("qtd_registros"),
        count("valor").alias("qtd_valores_numericos")
    )
    .orderBy("variavel", "produto")
)

display(comparacao_produtos)

### Definição das categorias utilizadas nas análises de café

Durante a preparação da camada Gold, foi realizada uma análise das categorias existentes na coluna `produto`, com atenção especial para `Total` e `Café (em grão) Total`.

A comparação mostrou que essas categorias não devem ser interpretadas como equivalentes. Para as variáveis `Quantidade produzida` e `Rendimento médio da produção`, a categoria `Total` possui 297 registros, porém nenhum apresenta valor numérico válido. Por outro lado, a categoria `Café (em grão) Total` possui valores numéricos para as 27 UFs e os 11 anos considerados na camada Gold.

Essa diferenciação não foi realizada na camada Silver porque seu objetivo foi estruturar, limpar e padronizar os dados provenientes da fonte, preservando as categorias originais disponibilizadas pelo IBGE. Assim, os valores de `produto`, incluindo `Total`, `Café (em grão) Total`, `Café (em grão) Arábica` e `Café (em grão) Canephora`, foram mantidos conforme apresentados na fonte para garantir rastreabilidade e evitar alterações semânticas indevidas.

A camada Gold, por sua vez, possui finalidade analítica. É nesta etapa que as categorias são selecionadas de acordo com cada pergunta de negócio. Portanto, não é necessário remover, substituir ou renomear a categoria `Total` na Silver, pois ela faz parte da estrutura original dos dados e pode possuir valores válidos para outras variáveis.

Para as análises referentes especificamente à produção de café, será utilizada a categoria `Café (em grão) Total`. Quando for necessária a análise dos diferentes tipos de café, serão utilizadas separadamente as categorias `Café (em grão) Arábica` e `Café (em grão) Canephora`.

Essa estratégia mantém a camada Silver fiel à estrutura tratada da fonte e utiliza a camada Gold para aplicar as regras e seleções necessárias ao contexto analítico, evitando também possíveis duplicidades na contabilização dos dados.

In [0]:
# Recorte da produção total de café para a Pergunta 1

df_producao_cafe = (
    df_gold_base
    .filter(
        (col("variavel") == "Quantidade produzida") &
        (col("produto") == "Café (em grão) Total")
    )
)

print("Quantidade de registros:", df_producao_cafe.count())

print(
    "Quantidade de UFs:",
    df_producao_cafe.select("uf").distinct().count()
)

print(
    "Quantidade de regiões:",
    df_producao_cafe.select("regiao").distinct().count()
)

print(
    "Quantidade de anos:",
    df_producao_cafe.select("ano").distinct().count()
)

print(
    "Registros com valor NULL:",
    df_producao_cafe.filter(col("valor").isNull()).count()
)

In [0]:
# Produção acumulada de café por UF entre 2015 e 2025

from pyspark.sql.functions import sum as spark_sum

producao_por_uf = (
    df_producao_cafe
    .groupBy("uf", "regiao")
    .agg(
        spark_sum("valor").alias("producao_total_toneladas")
    )
    .orderBy(
        col("producao_total_toneladas").desc()
    )
)

display(producao_por_uf)

In [0]:
# Produção acumulada de café por região entre 2015 e 2025

producao_por_regiao = (
    df_producao_cafe
    .groupBy("regiao")
    .agg(
        spark_sum("valor").alias("producao_total_toneladas")
    )
    .orderBy(
        col("producao_total_toneladas").desc()
    )
)

display(producao_por_regiao)

In [0]:
# Participação percentual de cada região na produção nacional de café

total_producao_brasil = (
    df_producao_cafe
    .agg(spark_sum("valor").alias("total"))
    .first()["total"]
)

producao_por_regiao_percentual = (
    producao_por_regiao
    .withColumn(
        "participacao_percentual",
        (col("producao_total_toneladas") / lit(total_producao_brasil)) * 100
    )
    .orderBy(col("producao_total_toneladas").desc())
)

print("Produção total considerada no período:", total_producao_brasil)

display(producao_por_regiao_percentual)

In [0]:
# Participação percentual dos estados na produção nacional de café

producao_por_uf_percentual = (
    producao_por_uf
    .withColumn(
        "participacao_percentual",
        (col("producao_total_toneladas") / lit(total_producao_brasil)) * 100
    )
    .orderBy(col("producao_total_toneladas").desc())
)

display(producao_por_uf_percentual)

In [0]:
# Gráfico das 10 UFs com maior produção acumulada de café (2015–2025)

top_10_ufs = (
    producao_por_uf_percentual
    .select(
        "uf",
        "producao_total_toneladas",
        "participacao_percentual"
    )
    .orderBy(col("producao_total_toneladas").desc())
    .limit(10)
)

display(top_10_ufs)

### Resposta à Pergunta 1 — Quais estados e regiões concentram a maior produção de café?

A análise da produção acumulada de café entre 2015 e 2025 evidencia uma forte concentração da produção brasileira na região Sudeste. No período analisado, o Sudeste acumulou aproximadamente 30,1 milhões de toneladas, correspondendo a cerca de 86% da produção considerada.

Entre as Unidades da Federação, Minas Gerais apresentou a maior produção acumulada, com aproximadamente 17,8 milhões de toneladas, representando cerca de 50,95% do total. Em seguida aparecem Espírito Santo, com aproximadamente 8,6 milhões de toneladas (24,71%), e São Paulo, com aproximadamente 3,4 milhões de toneladas (9,73%).

Fora da região Sudeste, destacam-se Bahia, com aproximadamente 2,36 milhões de toneladas, e Rondônia, com aproximadamente 1,57 milhão de toneladas.

Os resultados demonstram, portanto, que a produção de café está fortemente concentrada no Sudeste, principalmente em Minas Gerais e Espírito Santo, que juntos representam aproximadamente 75,7% da produção acumulada considerada no período analisado.

In [0]:
# Pergunta 2 - Evolução da produção total de café ao longo dos anos

producao_por_ano = (
    df_producao_cafe
    .groupBy("ano")
    .agg(
        spark_sum("valor").alias("producao_total_toneladas")
    )
    .orderBy("ano")
)

display(producao_por_ano)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, round

# Cálculo da variação percentual da produção em relação ao ano anterior

janela_ano = Window.orderBy("ano")

evolucao_producao = (
    producao_por_ano
    .withColumn(
        "producao_ano_anterior",
        lag("producao_total_toneladas").over(janela_ano)
    )
    .withColumn(
        "variacao_percentual",
        round(
            (
                (col("producao_total_toneladas") - col("producao_ano_anterior"))
                / col("producao_ano_anterior")
            ) * 100,
            2
        )
    )
)

display(evolucao_producao)

In [0]:
# Comparação entre o início e o final do período analisado

producao_2015 = (
    producao_por_ano
    .filter(col("ano") == 2015)
    .select("producao_total_toneladas")
    .first()[0]
)

producao_2025 = (
    producao_por_ano
    .filter(col("ano") == 2025)
    .select("producao_total_toneladas")
    .first()[0]
)

variacao_periodo = (
    (producao_2025 - producao_2015)
    / producao_2015
) * 100

print("Produção em 2015:", producao_2015)
print("Produção em 2025:", producao_2025)
print(f"Variação entre 2015 e 2025: {variacao_periodo:.2f}%")

### Pergunta 2 — Como a produção de café evoluiu ao longo do período analisado?

A análise da produção total de café entre 2015 e 2025 demonstra uma trajetória marcada por oscilações ao longo do período. A produção passou de aproximadamente 2,65 milhões de toneladas em 2015 para 3,48 milhões de toneladas em 2025, o que representa um crescimento acumulado de 31,61%.

Apesar do crescimento observado entre o início e o final da série, a evolução não ocorreu de forma contínua. Entre as principais variações anuais, destacam-se o crescimento de 32,35% em 2018, a redução de 15,21% em 2019, a recuperação de 23,04% em 2020 e uma nova queda de 19,43% em 2021.

O maior volume de produção do período foi registrado em 2020, com aproximadamente 3,71 milhões de toneladas. A partir de 2022, observa-se uma sequência de crescimento da produção, com variações positivas de 6,48% em 2022, 5,33% em 2023, 1,01% em 2024 e 3,01% em 2025.

Portanto, os dados indicam que, embora a produção de café tenha apresentado oscilações relevantes ao longo dos anos, o período analisado terminou com um volume de produção superior ao observado em 2015, evidenciando crescimento acumulado da produção no horizonte de 2015 a 2025.

In [0]:
# Pergunta 3 - Verificação dos dados de produtividade

dados_produtividade = (
    df_gold_base
    .filter(
        (col("variavel") == "Rendimento médio da produção") &
        (col("produto") == "Café (em grão) Total")
    )
)

print("Quantidade de registros:", dados_produtividade.count())
print("Quantidade de UFs:", dados_produtividade.select("uf").distinct().count())
print("Quantidade de anos:", dados_produtividade.select("ano").distinct().count())
print("Registros com valor NULL:", dados_produtividade.filter(col("valor").isNull()).count())

display(
    dados_produtividade
    .select("uf", "regiao", "ano", "valor")
    .orderBy("uf", "ano")
)

In [0]:
# Verificação de valores de produtividade iguais a zero

produtividade_zero = (
    dados_produtividade
    .filter(col("valor") == 0)
    .select("uf", "regiao", "ano", "valor")
    .orderBy("uf", "ano")
)

print("Quantidade de registros com produtividade igual a zero:",
      produtividade_zero.count())

print("Quantidade de UFs com produtividade igual a zero:",
      produtividade_zero.select("uf").distinct().count())

display(produtividade_zero)

In [0]:
# Quantidade de anos com produtividade igual a zero por UF

zeros_por_uf = (
    produtividade_zero
    .groupBy("uf", "regiao")
    .count()
    .withColumnRenamed("count", "qtd_anos_com_zero")
    .orderBy(col("qtd_anos_com_zero").desc(), "uf")
)

display(zeros_por_uf)

In [0]:
# Pergunta 3 - Produtividade média por UF entre 2015 e 2025

from pyspark.sql.functions import avg, round as spark_round

produtividade_media_uf = (
    dados_produtividade
    .groupBy("uf", "regiao")
    .agg(
        spark_round(
            avg("valor"), 2
        ).alias("produtividade_media_kg_ha")
    )
    .orderBy(
        col("produtividade_media_kg_ha").desc()
    )
)

display(produtividade_media_uf)

In [0]:
# Produtividade por UF no ano mais recente da série (2025)

produtividade_2025 = (
    dados_produtividade
    .filter(col("ano") == 2025)
    .select(
        "uf",
        "regiao",
        col("valor").alias("produtividade_2025_kg_ha")
    )
    .orderBy(
        col("produtividade_2025_kg_ha").desc()
    )
)

display(produtividade_2025)

In [0]:
# Verificação da produtividade do Rio Grande do Norte ao longo da série

display(
    dados_produtividade
    .filter(col("uf") == "Rio Grande do Norte")
    .select("ano", "valor")
    .orderBy("ano")
)

### Pergunta 3 — Quais estados apresentam maior produtividade?

Para avaliar a produtividade da cafeicultura, foi utilizada a variável **Rendimento médio da produção**, expressa em quilogramas por hectare (kg/ha). A análise considerou os dados anuais disponíveis entre 2015 e 2025.

Inicialmente, foram identificados registros com produtividade igual a zero. Esses valores foram mantidos na camada Silver, pois correspondem aos dados estruturados a partir da fonte original. Entretanto, para o cálculo da produtividade média por estado, os valores iguais a zero foram desconsiderados, evitando que anos sem produtividade registrada reduzissem artificialmente a média dos estados que possuem produção apenas em parte do período analisado.

Considerando apenas os anos com produtividade superior a zero, os estados com as maiores produtividades médias no período foram **Goiás (2.519,82 kg/ha)**, **Rondônia (2.486,09 kg/ha)**, **Acre (2.218,55 kg/ha)**, **Distrito Federal (2.186,18 kg/ha)** e **Espírito Santo (1.934,82 kg/ha)**.

Também foi realizada uma análise específica para 2025. Nesse ano, o **Rio Grande do Norte apresentou o maior rendimento, com 4.582 kg/ha**, seguido por **Acre (3.443 kg/ha)** e **Rondônia (3.354 kg/ha)**. No entanto, a análise histórica mostrou que o Rio Grande do Norte apresentou valores iguais a zero entre 2015 e 2022, passando a registrar rendimento positivo somente a partir de 2023. Por esse motivo, o resultado de 2025 deve ser interpretado como um desempenho recente, e não como indicação de liderança durante todo o período.

Assim, a análise evidencia que a produtividade pode variar significativamente entre os estados e ao longo dos anos, sendo importante diferenciar o **desempenho médio no período** do **resultado observado em um ano específico**.

In [0]:
# Pergunta 4 - Identificação dos principais estados produtores

principais_estados = (
    producao_por_uf
    .orderBy(col("producao_total_toneladas").desc())
    .limit(5)
)

display(principais_estados)

In [0]:
# Produção anual dos 5 principais estados produtores

lista_principais_estados = [
    "Minas Gerais",
    "Espírito Santo",
    "São Paulo",
    "Bahia",
    "Rondônia"
]

producao_anual_principais = (
    df_producao_cafe
    .filter(col("uf").isin(lista_principais_estados))
    .groupBy("ano", "uf")
    .agg(
        spark_sum("valor").alias("producao_estado_toneladas")
    )
    .orderBy("ano", col("producao_estado_toneladas").desc())
)

print("Quantidade de registros:", producao_anual_principais.count())

display(producao_anual_principais)

In [0]:
# Participação percentual anual dos principais estados produtores

participacao_anual_estados = (
    producao_anual_principais
    .join(
        producao_por_ano,
        on="ano",
        how="left"
    )
    .withColumn(
        "participacao_percentual",
        round(
            (col("producao_estado_toneladas") /
             col("producao_total_toneladas")) * 100,
            2
        )
    )
    .select(
        "ano",
        "uf",
        "producao_estado_toneladas",
        "producao_total_toneladas",
        "participacao_percentual"
    )
    .orderBy("ano", col("participacao_percentual").desc())
)

display(participacao_anual_estados)

In [0]:
# Comparação da participação dos principais estados entre 2015 e 2025

comparacao_participacao = (
    participacao_anual_estados
    .filter(col("ano").isin(2015, 2025))
    .select(
        "ano",
        "uf",
        "participacao_percentual"
    )
    .orderBy("uf", "ano")
)

display(comparacao_participacao)

In [0]:
# Variação da participação entre 2015 e 2025

from pyspark.sql.functions import first, col, round

participacao_2015_2025 = (
    comparacao_participacao
    .groupBy("uf")
    .pivot("ano", [2015, 2025])
    .agg(first("participacao_percentual"))
    .withColumn(
        "variacao_pontos_percentuais",
        round(col("2025") - col("2015"), 2)
    )
    .orderBy(col("variacao_pontos_percentuais").desc())
)

display(participacao_2015_2025)

### Pergunta 4 — Como a participação dos principais estados produtores mudou ao longo dos anos?

A análise da participação dos cinco principais estados produtores mostra mudanças na distribuição da produção brasileira de café entre 2015 e 2025.

Minas Gerais permaneceu como o principal estado produtor, porém sua participação na produção nacional caiu de 50,83% em 2015 para 44,89% em 2025, uma redução de 5,94 pontos percentuais.

O Espírito Santo apresentou o maior crescimento de participação entre os principais produtores, passando de 23,35% para 30,72%, um aumento de 7,37 pontos percentuais.

Rondônia também ampliou sua participação, de 3,20% para 4,18%, enquanto a Bahia apresentou relativa estabilidade, passando de 7,90% para 8,03%. São Paulo registrou uma pequena redução, de 9,61% para 9,08%.

Os resultados indicam que, apesar da manutenção da liderança de Minas Gerais, houve uma redistribuição da participação entre os principais estados produtores no período analisado, principalmente pelo crescimento da participação do Espírito Santo.

In [0]:
# Pergunta 5 - Verificação das categorias de café disponíveis

produtos_disponiveis = (
    spark.table("workspace.silver.ibge_pam_cafe")
    .filter(col("variavel") == "Quantidade produzida")
    .select("produto")
    .distinct()
    .orderBy("produto")
)

display(produtos_disponiveis)

In [0]:
# Pergunta 5 - Produção acumulada de Arábica e Canephora
# nos 5 principais estados produtores

producao_tipo_cafe = (
    spark.table("workspace.silver.ibge_pam_cafe")
    .filter(
        (col("variavel") == "Quantidade produzida") &
        (col("uf").isin(lista_principais_estados)) &
        (col("produto").isin(
            "Café (em grão) Arábica",
            "Café (em grão) Canephora"
        ))
    )
    .groupBy("uf", "produto")
    .agg(
        spark_sum("valor").alias("producao_total_toneladas")
    )
    .orderBy("uf", col("producao_total_toneladas").desc())
)

display(producao_tipo_cafe)

In [0]:
# Participação de Arábica e Canephora na produção de cada estado

from pyspark.sql.window import Window
from pyspark.sql.functions import sum as spark_sum, col, round

janela_estado = Window.partitionBy("uf")

distribuicao_tipo_cafe = (
    producao_tipo_cafe
    .withColumn(
        "total_estado",
        spark_sum("producao_total_toneladas").over(janela_estado)
    )
    .withColumn(
        "participacao_percentual",
        round(
            col("producao_total_toneladas") /
            col("total_estado") * 100,
            2
        )
    )
    .orderBy("uf", col("participacao_percentual").desc())
)

display(distribuicao_tipo_cafe)

### Pergunta 5 — Como a produção de café arábica e conilon se distribui entre os principais estados produtores?

A análise da produção acumulada entre 2015 e 2025 evidencia diferenças importantes na distribuição das variedades de café entre os cinco principais estados produtores. No conjunto de dados do IBGE, o café conilon é representado pela categoria Café (em grão) Canephora.

Minas Gerais apresenta forte predominância do café arábica, que corresponde a 98,75% de sua produção no período analisado. Situação semelhante ocorre em São Paulo, onde o arábica representa 99,97% da produção.

Em sentido oposto, Rondônia apresenta produção exclusivamente de Canephora nos dados analisados, correspondendo a 100% de sua produção. O Espírito Santo também apresenta predominância dessa variedade, com 75,33% de Canephora e 24,67% de arábica.

A Bahia apresenta uma distribuição mais equilibrada entre as duas variedades, com 57,00% de Canephora e 43,00% de arábica.

Os resultados demonstram uma especialização distinta entre os principais estados produtores: Minas Gerais e São Paulo apresentam forte concentração em café arábica, enquanto Rondônia e Espírito Santo possuem maior participação de Canephora. A Bahia apresenta a distribuição mais equilibrada entre as duas variedades.

In [0]:
# Verificação do schema Gold

spark.sql("SHOW TABLES IN workspace.gold").show(truncate=False)

In [0]:
# Gravação da tabela Gold - Produção anual de café

(
    producao_por_ano
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("workspace.gold.producao_anual_cafe")
)

print("Tabela workspace.gold.producao_anual_cafe criada com sucesso.")

In [0]:
# Validação da tabela Gold - Produção anual de café

df_gold_producao_anual = spark.table(
    "workspace.gold.producao_anual_cafe"
)

print("Quantidade de registros:", df_gold_producao_anual.count())

display(
    df_gold_producao_anual.orderBy("ano")
)

In [0]:
# Verificação das tabelas existentes na camada Gold

spark.sql("""
    SHOW TABLES IN workspace.gold
""").show(truncate=False)

In [0]:
# Validação dos principais estados produtores antes da gravação na Gold

print("Quantidade de estados:", principais_estados.count())

display(principais_estados)

In [0]:
# Gravação dos principais estados produtores na camada Gold

(
    principais_estados
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.principais_estados_produtores")
)

print("Tabela workspace.gold.principais_estados_produtores criada com sucesso.")

In [0]:
# Validação da tabela Gold - Principais estados produtores

df_gold_principais_estados = spark.table(
    "workspace.gold.principais_estados_produtores"
)

print("Quantidade de registros:", df_gold_principais_estados.count())

display(
    df_gold_principais_estados
    .orderBy(col("producao_total_toneladas").desc())
)

In [0]:
# Validação da participação anual dos principais estados

print(
    "Quantidade de registros:",
    participacao_anual_estados.count()
)

print(
    "Quantidade de anos:",
    participacao_anual_estados.select("ano").distinct().count()
)

print(
    "Quantidade de estados:",
    participacao_anual_estados.select("uf").distinct().count()
)

display(
    participacao_anual_estados.orderBy(
        "ano",
        col("participacao_percentual").desc()
    )
)

In [0]:
# Gravação da participação anual dos principais estados na camada Gold

(
    participacao_anual_estados
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.participacao_anual_estados")
)

print("Tabela workspace.gold.participacao_anual_estados criada com sucesso.")

In [0]:
# Validação da tabela Gold - Participação anual dos principais estados

df_gold_participacao = spark.table(
    "workspace.gold.participacao_anual_estados"
)

print("Quantidade de registros:", df_gold_participacao.count())

print(
    "Quantidade de anos:",
    df_gold_participacao.select("ano").distinct().count()
)

print(
    "Quantidade de estados:",
    df_gold_participacao.select("uf").distinct().count()
)

display(
    df_gold_participacao.orderBy(
        "ano",
        col("participacao_percentual").desc()
    )
)

In [0]:
# Verificação das tabelas existentes na camada Gold

spark.sql("""
    SHOW TABLES IN workspace.gold
""").show(truncate=False)

In [0]:
# Validação final das tabelas da camada Gold

print(
    "Produção anual:",
    spark.table("workspace.gold.producao_anual_cafe").count(),
    "registros"
)

print(
    "Principais estados produtores:",
    spark.table("workspace.gold.principais_estados_produtores").count(),
    "registros"
)

print(
    "Participação anual dos estados:",
    spark.table("workspace.gold.participacao_anual_estados").count(),
    "registros"
)

## Conclusão da Camada Gold

A camada Gold foi construída a partir dos dados tratados e estruturados na camada Silver, com o objetivo de disponibilizar informações agregadas e preparadas para análise.

Foram criadas três tabelas analíticas:

- **producao_anual_cafe**: apresenta a produção total de café por ano no período de 2015 a 2025.
- **principais_estados_produtores**: identifica os cinco estados com maior produção acumulada de café no período analisado.
- **participacao_anual_estados**: apresenta a produção e a participação percentual anual dos cinco principais estados produtores em relação à produção nacional.

As validações finais confirmaram 11 registros na tabela de produção anual, 5 registros na tabela dos principais estados produtores e 55 registros na tabela de participação anual dos estados.

Dessa forma, os dados percorrem as camadas Bronze, Silver e Gold, mantendo a separação entre dados brutos, dados tratados e dados preparados para consumo analítico.